In [ ]:
import os

In [ ]:
from PIL import Image

In [ ]:
import torch
# torch may be used, simply a placeholder model api until we decide

In [ ]:
import kagglehub, shutil

# Download latest version
path = kagglehub.dataset_download("payutch/fall-video-dataset")

print("Path to dataset files:", path)

# Define your target directory (relative to notebook)
target_dir = "./fall-video-dataset"

# Create directory if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

# Step 4: Copy dataset contents
for item in os.listdir(path):
    s = os.path.join(path, item)
    d = os.path.join(target_dir, item)
    
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("Dataset copied to:", target_dir)

Below is the short description as provided in the kaggle page we downloaded the dataset from Kaggle

```About this directory```

```-Raw_Video directory contains raw video footage of human fall.```

```-Keypoints_CSV directory```

```-For each input video (e.g., video1.mp4 in Fall/Raw_Video/), the script generates a corresponding CSV file (e.g., video1_keypoints.csv in Fall/Keypoints_CSV/).```

```-Each CSV file contains frame-by-frame data of the 17 extracted keypoints, their 2D coordinates, and confidence scores.```

```-The CSV file will have columns: "Frame", "Keypoint", "X", "Y", "Confidence".```





Referencing the dataset description on the kaggle page, we load the CSV into frame vectors

Load and Preprocess the CSV's


In [ ]:
import pandas as pd
import numpy as np 
from pathlib import Path

def load_video_keypoints(csv_path):
    df = pd.read_csv(csv_path)
    frames = sorted(df['Frame'].unique())
    keypoints_per_frame = []

    for frame in frames:
        frame_data = df[df['Frame'] == frame].sort_values('Keypoint')
        # Shape: (17, 3) — x, y, confidence
        kp = frame_data[['X', 'Y', 'Confidence']].values
        keypoints_per_frame.append(kp)

    return np.array(keypoints_per_frame)  # (T, 17, 3)

Build a Sliding Window Dataset, using 30 frame windows

In [ ]:
def create_sequences(keypoints, label, window_size=30, stride=15):
    sequences, labels = [], []
    T = len(keypoints)
    for start in range(0, T - window_size + 1, stride):
        seq = keypoints[start:start + window_size]  # (30, 17, 3)
        sequences.append(seq)
        labels.append(label)
    return sequences, labels

# Build full dataset
def build_dataset(fall_csv_dir, non_fall_csv_dir, window_size=30):
    X, y = [], []
    for path in Path(fall_csv_dir).glob("*.csv"):
        kp = load_video_keypoints(path)
        seqs, labs = create_sequences(kp, label=1, window_size=window_size)
        X.extend(seqs); y.extend(labs)

    for path in Path(non_fall_csv_dir).glob("*.csv"):
        kp = load_video_keypoints(path)
        seqs, labs = create_sequences(kp, label=0, window_size=window_size)
        X.extend(seqs); y.extend(labs)

    return np.array(X), np.array(y)  # (N, 30, 17, 3)

In [ ]:
X, y = build_dataset(
    fall_csv_dir = r"..\fall-video-dataset\Fall\Keypoints_CSV",
    non_fall_csv_dir = r"..\fall-video-dataset\No_Fall\Keypoints_CSV"
    )

Normalize the Data

In [ ]:
def normalize_keypoints(X):
    # X shape: (N, T, 17, 3)
    xy = X[..., :2]  # (N, T, 17, 2)
    conf = X[..., 2:3]

    min_xy = xy.min(axis=2, keepdims=True)  # per frame min
    max_xy = xy.max(axis=2, keepdims=True)
    range_xy = np.where((max_xy - min_xy) == 0, 1, max_xy - min_xy)

    xy_norm = (xy - min_xy) / range_xy  # → [0, 1]
    return np.concatenate([xy_norm, conf], axis=-1)

X, y = build_dataset("Fall/Keypoints_CSV", "NonFall/Keypoints_CSV")
X = normalize_keypoints(X)

Train/Validate/Split, so we dont have to make our own training dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Build CNN (Convolutional Neural Networks) + LSTM (Long Short-Term Memory networks) model 

In [ ]:
import tensorflow as tf
from keras import layers, models

def build_cnn_lstm(window_size=30, n_keypoints=17, n_features=3):
    inputs = tf.keras.Input(shape=(window_size, n_keypoints, n_features))

    # --- CNN block applied to each frame via TimeDistributed ---
    x = layers.TimeDistributed(layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling1D())(x)  # (batch, T, 128)

    # --- LSTM block for temporal modeling ---
    x = layers.LSTM(128, return_sequences=True, dropout=0.3)(x)
    x = layers.LSTM(64, dropout=0.3)(x)

    # --- Classifier head ---
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model

model = build_cnn_lstm()
model.summary()

Compile and Train the model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_auc', mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
    tf.keras.callbacks.ModelCheckpoint("best_fall_detection_model.keras", save_best_only=True, monitor='val_auc', mode='max')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

Evaulate the model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=['Non-Fall', 'Fall']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Non-Fall','Fall'], yticklabels=['Non-Fall','Fall'])
plt.title("Confusion Matrix"); plt.show()